## Using PulseKit to run PTT pipeline

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PulseKit import *  
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
import os

#### Load Dataset, Get Cardiac Cycles, and process signals

In [2]:
dataset = Dataset(
    datasetName="TestDataset",
    sessionIds=["52", "53", "54", "55", "56", "57"],
    signals=[
        ECG(name="ECG"),
        PPG(name="PPG_Finger"),
        PPG(name="PPG_Ear"),
        Stethoscope(name="Digital_Stethoscope"),
        Earbud(name="Earbud_Left"),
        Earbud(name="Earbud_Right"),
        Accelerometer(name="Accelerometer_X"),
        Accelerometer(name="Accelerometer_Y"),
        Accelerometer(name="Accelerometer_Z"),
        RespirationBelt(name="Respiration_Belt"),
        BP(name="Diastolic_BP"),
        BP(name="Systolic_BP"),
        BP(name="NIBP"),
        Hemodynamics(name="Cardiac_Output"),
        Hemodynamics(name="Stroke_Volume"),
        Hemodynamics(name="SVR"),
        Hemodynamics(name="Heart_Rate")
    ],
    data_directory="test_data"
)

In [3]:
dataset.get_cardiac_cycles(reference_signal_name="ECG")

In [4]:
dataset.process_signals(signal_list=["ECG", "PPG_Finger", "PPG_Ear", "Earbud_Left", "Systolic_BP", "Diastolic_BP"])

Processing session 52...
Processing session 53...
Processing session 54...
Processing session 55...
Processing session 56...
Processing session 57...


#### Compute features and populate tables with targets

In [5]:
dataset.compute_features(
    features=[
        PWA(dataset=dataset, signals=["PPG_Finger", "PPG_Ear"]),
        PTT(dataset=dataset, signals=["PPG_Finger", "Earbud_Left", "ECG"])
    ],
    targets=["Systolic_BP", "Diastolic_BP"],
    output_name="test_features",
    activity_labels=True
)

Saved to: test_data/test_features.csv


#### Run Models: e.g. Linear PTT Model for Systolic BP

In [10]:
CALIBRATION_N = 32 
df = pd.read_csv("test_data/test_features.csv")
results = []
np.random.seed(42)


# Keep only relevant columns and drop rows where quality check failed (NaNs)
subset = df[["participant_id", "session_id", "PAT_PPG_Finger_downsample", "Systolic_BP_downsample_midpoint"]].copy()
subset.dropna(inplace=True)
    
# Subject specific modeling
for subject_id in subset["participant_id"].unique():
    subject_data = subset[subset["participant_id"] == subject_id]
        
    X = subject_data["PAT_PPG_Finger_downsample"].values.reshape(-1, 1)
    y = subject_data["Systolic_BP_downsample_midpoint"].values
    
    # Select Calibration Points (Stratified: Low + High values)
    n_half = CALIBRATION_N // 2
    train_idx_low = np.random.choice(np.where(y < np.percentile(y, 50))[0], n_half, replace=False)
    train_idx_high = np.random.choice(np.where(y > np.percentile(y, 50))[0], n_half, replace=False)
    train_idx = np.concatenate([train_idx_low, train_idx_high])
    
    # Data split
    X_train, y_train = X[train_idx], y[train_idx]
    mask = np.ones(len(y), dtype=bool)
    mask[train_idx] = False
    X_test, y_test = X[mask], y[mask]

    # Run model and eval
    model = LinearRegression()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    results.append({
        "subject_id": subject_id,
        "calibration_points": CALIBRATION_N,
        "total_samples": len(subject_data),
        "MAE": round(mae, 3),
        "R2": round(r2, 3),
        "Slope (beta)": round(model.coef_[0], 4),
        "Intercept (alpha)": round(model.intercept_, 4)
    })

final_df = pd.DataFrame(results)
final_df.head()

,subject_id,calibration_points,total_samples,MAE,R2,Slope (beta),Intercept (alpha)
0,3,32,2091,14.469,-0.187,-0.2003,151.9753
1,41,32,1986,11.737,0.095,-0.3040,203.2458
